# CVE-2026-31431 — OpenClaw Exploitation Investigation

**Linux kernel `algif_aead` local privilege escalation**

This notebook:
1. Loads telemetry from a **local JSON file** or **S3 bucket** (`--raw-telemetry` flag)
2. Runs deterministic signal queries Q2–Q6 via pandas
3. Scores signals against the CVE-2026-31431 rule set (Tier 1)
4. Invokes **OpenClaw** as a subprocess for LLM reasoning (Tier 2)
5. Emits a **YAML finding** and **Markdown report** for GitHub Pages

Run headlessly: `papermill notebooks/investigate_CVE-2026-31431.ipynb out.ipynb -p raw_telemetry s3`

In [ ]:
# ============================================================
# PARAMETERS — injected by papermill at runtime
# All values below are defaults (used when running interactively)
# ============================================================
raw_telemetry         = "local"                            # "local" | "s3"
cve_id                = "CVE-2026-31431"
workload_id           = "ALL"                              # "ALL" or specific wl_id
output_dir            = "../findings"
local_events_path     = "../data/sample_events.json"       # swap to events.json in prod
local_inventory_path  = "../data/sample_inventory.csv"    # swap to inventory.csv in prod
s3_manifest_path      = "../data/s3_manifest.json"
lookback_hours        = 24
run_id                = None                               # auto-generated if None
openclaw_bin          = "../openclaw"
signals_file          = "../signals/CVE-2026-31431.yaml"

In [ ]:
# ============================================================
# Cell 2 — Imports & Setup
# ============================================================
import json
import subprocess
import uuid
import pathlib
import datetime
import sys

import pandas as pd
import yaml
from IPython.display import display, Markdown, HTML

# Auto-generate run_id if not injected
if run_id is None:
    run_id = str(uuid.uuid4())

output_path = pathlib.Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

# Cutoff timestamp for event filtering
cutoff_ts = (
    datetime.datetime.utcnow() - datetime.timedelta(hours=lookback_hours)
).timestamp()

display(Markdown(f"""
## Investigation Setup
| Field | Value |
|---|---|
| **CVE** | `{cve_id}` |
| **Run ID** | `{run_id}` |
| **Telemetry source** | `{raw_telemetry}` |
| **Workload filter** | `{workload_id}` |
| **Lookback** | {lookback_hours} hours |
"""))

In [ ]:
# ============================================================
# Cell 3 — Data Fetch: Local JSON or S3 (Named Profile)
# ============================================================
import boto3
import io

if raw_telemetry == "local":
    display(Markdown(f"**Telemetry source**: local file `{local_events_path}`"))

    with open(local_events_path, "r") as f:
        events_raw = json.load(f)
    events_df = pd.DataFrame(events_raw)
    inventory_df = pd.read_csv(local_inventory_path, dtype=str)

elif raw_telemetry == "s3":
    manifest = json.loads(pathlib.Path(s3_manifest_path).read_text())
    bucket   = manifest["bucket"]
    region   = manifest["region"]
    profile  = manifest.get("profile_name", "default")  # named profile from manifest

    display(Markdown(
        f"**Telemetry source**: S3 `s3://{bucket}/{manifest['telemetry_key']}`  \n"
        f"**AWS profile**: `{profile}` (from `~/.aws/credentials`)"
    ))

    # boto3 named-profile session — no secrets in code
    session = boto3.Session(profile_name=profile, region_name=region)
    s3 = session.client("s3")

    # Download telemetry JSON
    tel_obj = s3.get_object(Bucket=bucket, Key=manifest["telemetry_key"])
    events_df = pd.read_json(io.BytesIO(tel_obj["Body"].read()))

    # Download inventory CSV
    inv_obj = s3.get_object(Bucket=bucket, Key=manifest["inventory_key"])
    inventory_df = pd.read_csv(io.BytesIO(inv_obj["Body"].read()), dtype=str)

    # Optionally verify integrity
    try:
        checksum_key = manifest["telemetry_key"] + ".sha256"
        chk_obj = s3.get_object(Bucket=bucket, Key=checksum_key)
        expected_hash = chk_obj["Body"].read().decode().strip()
        display(Markdown(f"**Checksum on file**: `{expected_hash[:16]}...` (verify locally if needed)"))
    except Exception:
        display(Markdown("_No checksum file found — skipping integrity check._"))

else:
    raise ValueError(f"Unknown raw_telemetry value: {raw_telemetry!r}. Use 'local' or 's3'.")

# ---- Filter by workload_id ----
if workload_id != "ALL" and "workload_id" in events_df.columns:
    events_df = events_df[events_df["workload_id"] == workload_id]

# ---- Apply lookback window ----
if "ts" in events_df.columns:
    events_df = events_df[events_df["ts"] >= cutoff_ts]

# Normalise types
for col in ["uid", "euid", "socket_family", "socket_protocol", "pid", "ppid"]:
    if col in events_df.columns:
        events_df[col] = pd.to_numeric(events_df[col], errors="coerce")

print(f"Events loaded: {len(events_df):,} rows")
print(f"Inventory loaded: {len(inventory_df):,} workloads")
display(events_df.dtypes.to_frame("dtype").T)

In [ ]:
# ============================================================
# Cell 4 — Inventory Match (Q1: algif_aead loaded?)
# Identifies workloads that require deeper telemetry investigation
# ============================================================
TRIGGER_ASSESSMENTS = [
    "vulnerable_or_not_confirmed_fixed",
    "running_kernel_pkg_not_matched",
]

target_workloads = inventory_df[
    inventory_df["assessment"].isin(TRIGGER_ASSESSMENTS)
].copy()

algif_loaded_hosts = inventory_df[
    inventory_df.get("algif_aead", pd.Series(dtype=str)) == "loaded"
]

algif_loaded_signal = len(algif_loaded_hosts) > 0

display(Markdown("### Q1 — Inventory: Workloads Requiring Investigation"))
display(Markdown(
    f"**Trigger workloads found**: {len(target_workloads)}  \n"
    f"**algif_aead loaded on at least one host**: {'✅ YES' if algif_loaded_signal else '—'}"
))
if not target_workloads.empty:
    display(target_workloads[
        [c for c in ["hostname", "distro_family", "running_kernel", "algif_aead", "assessment"]
         if c in target_workloads.columns]
    ])

In [ ]:
# ============================================================
# Cell 5 — Q2: AF_ALG Socket Opens by Unprivileged Processes
# Socket family 38 = AF_ALG (Linux crypto interface)
# Signal: Non-root process opening an AF_ALG socket is suspicious
# ============================================================
q2 = events_df[
    (events_df.get("socket_family", pd.Series(dtype=float)) == 38) &
    (events_df.get("uid", pd.Series(dtype=float)) > 0)
][[c for c in [
    "workload_id", "hostname", "pid", "uid", "euid",
    "process_name", "cmdline", "socket_protocol", "ts"
] if c in events_df.columns]].sort_values("ts", ascending=False)

q2_signal = len(q2) > 0

display(Markdown(
    f"### Q2 — Unprivileged AF_ALG Socket Opens: **{len(q2)} event(s)**  \n"
    + ("🟠 **SIGNAL DETECTED**" if q2_signal else "✅ None found")
))
if not q2.empty:
    display(q2)

In [ ]:
# ============================================================
# Cell 6 — Q3: UID Escalation After AF_ALG Socket Open
# PRIMARY EXPLOITATION SIGNAL for CVE-2026-31431
# A process that opened AF_ALG as uid>0 and then achieved euid=0
# ============================================================
q3 = events_df[
    (events_df.get("socket_family", pd.Series(dtype=float)) == 38) &
    (events_df.get("uid", pd.Series(dtype=float)) != 0) &
    (events_df.get("euid", pd.Series(dtype=float)) == 0)
][[c for c in [
    "workload_id", "hostname", "pid", "ppid",
    "uid", "euid", "process_name", "cmdline", "ts"
] if c in events_df.columns]].sort_values("ts", ascending=False)

q3_signal = len(q3) > 0

display(Markdown(
    f"### Q3 — UID Escalation via AF_ALG ← **PRIMARY SIGNAL**: **{len(q3)} event(s)**  \n"
    + ("🔴 **EXPLOITATION INDICATOR FOUND**" if q3_signal else "✅ None found")
))
if not q3.empty:
    display(
        q3.style.applymap(
            lambda _: "background-color:#ff4444;color:white",
            subset=[c for c in ["euid", "uid"] if c in q3.columns]
        )
    )

In [ ]:
# ============================================================
# Cell 7 — Q4: Root Shell Spawned from Unprivileged Parent
# Post-exploitation signal: attacker spawns a root shell
# ============================================================
SHELL_NAMES = frozenset([
    "bash", "sh", "dash", "su", "sudo",
    "id", "whoami", "passwd", "useradd", "newgrp"
])

proc_events = events_df[events_df.get("event_type", pd.Series(dtype=str)) == "process"].copy()

if not proc_events.empty and "process_name" in proc_events.columns:
    # Children: root shells
    root_shells = proc_events[
        (proc_events["process_name"].isin(SHELL_NAMES)) &
        (proc_events.get("euid", pd.Series(dtype=float)) == 0)
    ].copy()

    # Parents: non-root processes
    parent_map = proc_events.set_index("pid")[["uid", "cmdline"]].rename(
        columns={"uid": "parent_uid", "cmdline": "parent_cmdline"}
    )

    if not root_shells.empty and "ppid" in root_shells.columns:
        q4 = root_shells.join(parent_map, on="ppid", how="left")
        q4 = q4[q4["parent_uid"].notna() & (q4["parent_uid"] != 0)]
    else:
        q4 = pd.DataFrame()
else:
    q4 = pd.DataFrame()

q4_signal = len(q4) > 0

display(Markdown(
    f"### Q4 — Root Shell from Non-Root Parent: **{len(q4)} event(s)**  \n"
    + ("🔴 **POST-EXPLOITATION ACTIVITY**" if q4_signal else "✅ None found")
))
if not q4.empty:
    display(q4)

In [ ]:
# ============================================================
# Cell 8 — Q5: algif_aead Module Load Events
# Exploit may explicitly load the module via socket() call
# ============================================================
q5 = events_df[
    (events_df.get("event_type", pd.Series(dtype=str)) == "kernel_module") &
    (events_df.get("module_name", pd.Series(dtype=str)) == "algif_aead")
][[c for c in [
    "workload_id", "hostname", "ts", "module_name", "cmdline", "pid", "uid"
] if c in events_df.columns]].sort_values("ts", ascending=False)

q5_signal = len(q5) > 0

display(Markdown(
    f"### Q5 — algif_aead Module Load Events: **{len(q5)} event(s)**  \n"
    + ("🟠 **MODULE LOAD DETECTED**" if q5_signal else "✅ None found")
))
if not q5.empty:
    display(q5)

In [ ]:
# ============================================================
# Cell 9 — Q6: Exploit Staging Artifacts
# Common exploit delivery: /tmp, /dev/shm, memfd_create
# ============================================================
STAGING_PREFIXES = ("/tmp/", "/dev/shm/", "/proc/")

file_events = events_df[events_df.get("event_type", pd.Series(dtype=str)) == "file"].copy()

if not file_events.empty and "file_path" in file_events.columns:
    q6 = file_events[
        file_events["file_path"].str.startswith(STAGING_PREFIXES, na=False) |
        file_events.get("cmdline", pd.Series(dtype=str)).str.contains("memfd", na=False)
    ][[c for c in [
        "workload_id", "hostname", "pid", "uid", "euid",
        "file_path", "cmdline", "ts"
    ] if c in file_events.columns]].sort_values("ts", ascending=False)
else:
    q6 = pd.DataFrame()

q6_signal = len(q6) > 0

display(Markdown(
    f"### Q6 — Exploit Staging Artifacts: **{len(q6)} event(s)**  \n"
    + ("🟠 **STAGING ARTIFACTS FOUND**" if q6_signal else "✅ None found")
))
if not q6.empty:
    display(q6)

In [ ]:
# ============================================================
# Cell 10 — Tier 1 Signal Scoring
# Deterministic verdict from weighted rule set
# ============================================================
signals = {
    "ALGIF_LOADED": {
        "fired":  algif_loaded_signal,
        "weight": 0.3,
        "tier":   "suspicious",
        "q": "Q1",
    },
    "AF_ALG_SOCKET_OPEN_UNPRIV": {
        "fired":  q2_signal,
        "weight": 0.5,
        "tier":   "suspicious",
        "q": "Q2",
    },
    "UID_ESCALATION_AFTER_AFALG": {
        "fired":  q3_signal,
        "weight": 1.0,
        "tier":   "exploited",  # PRIMARY signal
        "q": "Q3",
    },
    "ROOT_SHELL_FROM_UNPRIV_PARENT": {
        "fired":  q4_signal,
        "weight": 0.9,
        "tier":   "exploited",
        "q": "Q4",
    },
    "MODULE_LOAD_EVENT": {
        "fired":  q5_signal,
        "weight": 0.4,
        "tier":   "suspicious",
        "q": "Q5",
    },
    "EXPLOIT_STAGING_ARTIFACTS": {
        "fired":  q6_signal,
        "weight": 0.6,
        "tier":   "suspicious",
        "q": "Q6",
    },
}

total_weight    = sum(v["weight"] for v in signals.values() if v["fired"])
signals_fired   = [k for k, v in signals.items() if v["fired"]]
exploited_fired = any(v["fired"] and v["tier"] == "exploited" for v in signals.values())

if exploited_fired and total_weight >= 1.0:
    tier1_verdict = "exploited"
elif total_weight >= 0.5:
    tier1_verdict = "suspicious"
elif total_weight == 0.0:
    tier1_verdict = "benign"
else:
    tier1_verdict = "inconclusive"

VERDICT_COLOR = {
    "exploited":    "#ff4444",
    "suspicious":   "#ff8c00",
    "benign":       "#22aa44",
    "inconclusive": "#888888",
}

display(HTML(f"""
<div style="border:2px solid {VERDICT_COLOR[tier1_verdict]};border-radius:8px;
            padding:16px;margin:12px 0;background:{VERDICT_COLOR[tier1_verdict]}22">
  <h2 style="color:{VERDICT_COLOR[tier1_verdict]};margin:0">
    Tier 1 Verdict: {tier1_verdict.upper()}
  </h2>
  <p><b>Total weight:</b> {total_weight:.2f}</p>
  <p><b>Signals fired:</b> {', '.join(signals_fired) or 'none'}</p>
</div>
"""))

# Print signal table
sig_rows = []
for sig_id, s in signals.items():
    sig_rows.append({
        "Rule": sig_id, "Query": s["q"],
        "Fired": "✅" if s["fired"] else "—",
        "Weight": s["weight"], "Tier": s["tier"]
    })
display(pd.DataFrame(sig_rows).set_index("Rule"))

In [ ]:
# ============================================================
# Cell 11 — OpenClaw Subprocess (Tier 2 LLM Reasoning)
# Passes Tier 1 results to OpenClaw for deeper analysis
# ============================================================
tier1_results = {
    "run_id":          run_id,
    "cve_id":          cve_id,
    "raw_telemetry":   raw_telemetry,
    "tier1_verdict":   tier1_verdict,
    "signals":         signals,
    "signals_fired":   signals_fired,
    "total_weight":    total_weight,
    "q2_rows":         q2.to_dict("records") if not q2.empty else [],
    "q3_rows":         q3.to_dict("records") if not q3.empty else [],
    "q4_rows":         q4.to_dict("records") if not q4.empty else [],
    "q5_rows":         q5.to_dict("records") if not q5.empty else [],
    "q6_rows":         q6.to_dict("records") if not q6.empty else [],
    "workloads":       target_workloads.to_dict("records") if not target_workloads.empty else [],
}

tier1_path = output_path / f"{run_id}_tier1.json"
tier1_path.write_text(json.dumps(tier1_results, default=str, indent=2))
print(f"Tier 1 results written: {tier1_path}")

# Invoke OpenClaw
openclaw_path = pathlib.Path(openclaw_bin)
openclaw_output_path = output_path / f"{run_id}_openclaw.json"
openclaw_finding = {}

if openclaw_path.exists():
    display(Markdown(f"**Invoking OpenClaw**: `{openclaw_bin}`"))
    result = subprocess.run(
        [
            str(openclaw_path), "investigate",
            "--cve",            cve_id,
            "--workload-id",    workload_id,
            "--tier1-results",  str(tier1_path),
            "--exploit-signals", signals_file,
            "--raw-telemetry",  raw_telemetry,
            "--output-json",
            "--output-file",    str(openclaw_output_path),
        ],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if result.returncode == 0:
        try:
            openclaw_finding = json.loads(result.stdout) if result.stdout.strip() else {}
            if openclaw_output_path.exists():
                openclaw_finding = json.loads(openclaw_output_path.read_text())
            display(Markdown(f"**OpenClaw verdict**: `{openclaw_finding.get('verdict', 'n/a')}`"))
        except json.JSONDecodeError:
            display(Markdown("⚠️ OpenClaw returned non-JSON output — see raw below"))
            print(result.stdout[:2000])
    else:
        display(Markdown(f"⚠️ OpenClaw exited with code {result.returncode}"))
        print("STDERR:", result.stderr[:1000])
else:
    display(Markdown(
        f"⚠️ OpenClaw binary not found at `{openclaw_bin}`.  \n"
        "Tier 2 reasoning skipped — Tier 1 verdict is the final result."
    ))
    openclaw_finding = {
        "verdict": tier1_verdict,
        "confidence": round(min(total_weight, 1.0), 2),
        "reasoning_trace": (
            f"OpenClaw binary not available. Tier 1 deterministic verdict: {tier1_verdict}. "
            f"Signals fired: {', '.join(signals_fired) or 'none'}. "
            f"Total signal weight: {total_weight:.2f}."
        ),
        "recommended_action": (
            "IMMEDIATE: Isolate affected workloads and patch kernel." if tier1_verdict == "exploited"
            else "Review signals and apply kernel patch or kmod mitigation."
        ),
    }

In [ ]:
# ============================================================
# Cell 12 — Output: YAML Finding
# Machine-readable finding with full evidence
# ============================================================
finding = {
    "schema_version":        "1.0",
    "run_id":                run_id,
    "cve_id":                cve_id,
    "raw_telemetry_source":  raw_telemetry,
    "investigated_at":       datetime.datetime.utcnow().isoformat() + "Z",
    "workloads_investigated": target_workloads["hostname"].tolist() if not target_workloads.empty else [],
    "tier1": {
        "verdict":       tier1_verdict,
        "signals_fired": signals_fired,
        "total_weight":  round(total_weight, 2),
    },
    "tier2":                openclaw_finding,
    "recommended_action":   openclaw_finding.get(
        "recommended_action", "Manual review required."
    ),
    # Private fields used by finding_store for richer Markdown rendering
    "_all_signals":         signals,
    "_workload_rows":       target_workloads.to_dict("records") if not target_workloads.empty else [],
    "_evidence": {
        "q2_rows": q2.head(20).to_dict("records") if not q2.empty else [],
        "q3_rows": q3.head(20).to_dict("records") if not q3.empty else [],
        "q4_rows": q4.head(20).to_dict("records") if not q4.empty else [],
        "q5_rows": q5.head(20).to_dict("records") if not q5.empty else [],
        "q6_rows": q6.head(20).to_dict("records") if not q6.empty else [],
    },
}

# Strip private fields for YAML export
yaml_finding = {k: v for k, v in finding.items() if not k.startswith("_")}
yaml_path = output_path / f"{run_id}_finding.yaml"
yaml_path.write_text(
    yaml.dump(yaml_finding, default_flow_style=False, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)
print(f"YAML finding written: {yaml_path}")
display(Markdown(f"```yaml\n{yaml_path.read_text()[:1500]}\n```"))

In [ ]:
# ============================================================
# Cell 13 — Output: Markdown Report for GitHub Pages
# Jekyll front-matter + tables + evidence + reasoning
# ============================================================
VERDICT_ICON = {
    "exploited":    "🔴",
    "suspicious":   "🟠",
    "benign":       "✅",
    "inconclusive": "⚪",
}
icon = VERDICT_ICON.get(tier1_verdict, "⚪")
date_str = datetime.datetime.utcnow().strftime("%Y-%m-%d")

def df_to_md(df, max_rows=20):
    if df.empty:
        return "_No events found._"
    try:
        return df.head(max_rows).to_markdown(index=False)
    except Exception:
        return df.head(max_rows).to_string(index=False)

md_lines = [
    "---",
    f'title: "{cve_id} Investigation — {run_id[:8]}"',
    f"date: {date_str}",
    f"cve: {cve_id}",
    f"verdict: {tier1_verdict}",
    f"run_id: {run_id}",
    "layout: finding",
    "---",
    "",
    f"# {cve_id} — Exploitation Investigation Report",
    "",
    "| Field | Value |",
    "|---|---|",
    f"| **Run ID** | `{run_id}` |",
    f"| **Investigated at** | {finding['investigated_at']} |",
    f"| **Telemetry source** | `{raw_telemetry}` |",
    f"| **Verdict** | {icon} **{tier1_verdict.upper()}** |",
    f"| **Total signal weight** | `{round(total_weight, 2)}` |",
    "",
    "---",
    "",
    "## Workloads Investigated",
    "",
]

if not target_workloads.empty:
    cols = [c for c in ["hostname", "distro_family", "running_kernel", "algif_aead", "assessment"]
            if c in target_workloads.columns]
    md_lines.append(df_to_md(target_workloads[cols]))
else:
    md_lines.append("_No vulnerable workloads identified._")

md_lines += [
    "",
    "---",
    "",
    "## Signal Analysis",
    "",
    "| Rule | Query | Fired | Weight | Tier |",
    "|---|---|---|---|---|",
]
for sig_id, s in signals.items():
    fired = "✅" if s["fired"] else "—"
    md_lines.append(f"| `{sig_id}` | {s['q']} | {fired} | {s['weight']} | {s['tier']} |")

md_lines += [
    "",
    f"**Signals fired**: {', '.join(f'`{s}`' for s in signals_fired) or '_none_'}",
    "",
    "---",
    "",
    "## Evidence",
    "",
    f"### Q3 — UID Escalation (Primary Signal): {len(q3)} event(s)",
    "",
    df_to_md(q3),
    "",
    f"### Q2 — AF_ALG Socket Opens: {len(q2)} event(s)",
    "",
    df_to_md(q2),
    "",
    f"### Q4 — Root Shell from Unprivileged Parent: {len(q4)} event(s)",
    "",
    df_to_md(q4),
    "",
    f"### Q5 — algif_aead Module Load Events: {len(q5)} event(s)",
    "",
    df_to_md(q5),
    "",
    f"### Q6 — Exploit Staging Artifacts: {len(q6)} event(s)",
    "",
    df_to_md(q6),
    "",
    "---",
    "",
    "## OpenClaw Reasoning (Tier 2)",
    "",
    openclaw_finding.get("reasoning_trace", "_OpenClaw not invoked or produced no output._"),
    "",
    "---",
    "",
    "## Recommended Action",
    "",
    f"> ⚠️ {finding['recommended_action']}",
    "",
    "---",
    f"*Generated by OpenClaw Harness v1.0.0 | {finding['investigated_at']}*",
]

slug     = f"{cve_id}_{run_id[:8]}"
docs_dir = pathlib.Path(output_dir).parent / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)
md_path  = docs_dir / f"{slug}.md"
md_path.write_text("\n".join(md_lines), encoding="utf-8")
print(f"Markdown report written: {md_path}")

display(Markdown(f"""
## ✅ Investigation Complete
| Output | Path |
|---|---|
| YAML finding | `{yaml_path}` |
| Markdown report | `{md_path}` |
| Tier 1 JSON | `{tier1_path}` |
"""))